# Schuerch - CODEX Colorectal cancer

Schürch, C. M. et al. Coordinated Cellular Neighborhoods Orchestrate Antitumoral Immunity at the Colorectal Cancer Invasive Front. Cell 182, 1341–1359.e19 (2020).

In [9]:
import scanpy as sc
import numpy as np
import squidpy as sq
import anndata as ad

In [10]:
from graph_transformer_long_range_niches._paths import CFG_FILES, RESULTS, FIG_PATH
from graph_transformer_long_range_niches.pp import split_adata

from InterScale.tl import split_adata_patient_stratified

## Single image

In [3]:
MICHAEL_MO_PATH = '/lustre/groups/ml01/datasets/projects/2024_Multiplexformer_pretrain_michael.dammann/MultiplexformerPretrainingData/anndata_groupby/4_schuerch_crc/'

In [4]:
adata = sc.read_h5ad(f'{MICHAEL_MO_PATH}/4_CRC_clusters_neighborhoods_markers_reg001_A.h5ad')

In [5]:
adata

AnnData object with n_obs × n_vars = 1164 × 56
    obs: 'CellType', 'X', 'Y', 'neighborhood number final', 'neighborhood name', 'TMA_AB', 'groups', 'patients'
    var: 'protein_names'

- `groups`: ‘Crohn’s-like reaction’ (CLR) and diffuse inflammatory infiltration (DII)
- `patients`: Total 35, each patient has 4 images
- `TMA_AB`: tissue microarrays: A, B

In [6]:
adata.var_names

Index(['CD44', 'FOXP3', 'CD8', 'p53', 'GATA3', 'CD45', 'T-bet', 'B-catenin',
       'HLA-DR', 'PD-L1', 'Ki67', 'CD45RA', 'CD4', 'CD21', 'MUC-1', 'CD30',
       'CD2', 'vimentin', 'CD20', 'LAG-3', 'NaKATPase', 'CD5', 'IDO-1',
       'cytokeratin', 'CD11b', 'CD56', 'SMA', 'BCL2', 'CD25', 'CD11c', 'PD-1',
       'GranzymeB', 'EGFR', 'VISTA', 'CD15', 'ICOS', 'Synaptophysin', 'GFAP',
       'CD7', 'CD3', 'Chromogranin A', 'CD163', 'CD45RO', 'CD68', 'CD31',
       'podoplanin', 'CD34', 'CD38', 'CD138', 'CDX2', 'collagenIV', 'CD194',
       'MMP9', 'CD71', 'CD57', 'MMP12'],
      dtype='object')

In [7]:
len(adata.obs['X'])

1164

In [8]:
(adata.obs['X'], adata.obs['Y'])).T

SyntaxError: unmatched ')' (3535949783.py, line 1)

In [ ]:
adata.obs['CellType']

In [ ]:
sq.gr.spatial_neighbors(adata)

In [ ]:
sq.pl.spatial_scatter(adata, 
                      color=["CellType", 'CD44', 'CD8'], 
                      shape=None,
                     size=20,
                     #figsize=(20, 5)
                     )

## All images

In [ ]:
len(adata.obs_names)

In [ ]:
img = 0
adatas = []
for TMA in ['A', 'B']:
    for i in range (1,71):
        adata = sc.read_h5ad(f'{MICHAEL_MO_PATH}/4_CRC_clusters_neighborhoods_markers_reg{i:03}_{TMA}.h5ad')
        adata.obs['image'] = np.repeat(img, len(adata.obs_names))
        img += 1
        adatas.append(adata)

In [ ]:
ad.concat(adatas, index_unique="_").obs

In [ ]:
adata_full = ad.concat(adatas, merge="same")
adata_full

Load concatinated file

In [ ]:
np.unique(adata_full.obs['patients'])

In [ ]:
np.unique(adata_full.obs['image']), len(np.unique(adata_full.obs['image']))

In [ ]:
adata

In [ ]:
adata_full = split_adata_patient_stratified(
    adata=adata_full,
    patient_obs='patients',      # Column containing patient IDs
    stratify_obs='groups',      # Column to stratify by (e.g., disease status)
    val_size=0.1,                  # 10% of patients for validation
    test_size=0.2,                 # 20% of patients for testing
    seed=42
)

In [ ]:
adata_full

In [ ]:
adata_full.obsm['spatial'] = np.array((adata_full.obs['X'], adata_full.obs['Y'])).T

In [ ]:
sq.gr.spatial_neighbors(adata_full, radius = 100)

In [ ]:
A = adata.obsp['spatial_connectivities']

In [ ]:
A = adata.obsp['spatial_connectivities']
degrees = A.sum(axis=1)

# Average connectivity
average_connectivity = degrees.mean()
average_connectivity

In [ ]:
adata_full.write('/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/data/schuerch20.h5ad')

In [12]:
adata = sc.read_h5ad('/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/data/schuerch20.h5ad')
adata

AnnData object with n_obs × n_vars = 258385 × 56
    obs: 'CellType', 'X', 'Y', 'neighborhood number final', 'neighborhood name', 'TMA_AB', 'groups', 'patients', 'image', 'split'
    var: 'protein_names'
    uns: 'spatial_neighbors'
    obsm: 'spatial'
    obsp: 'spatial_connectivities', 'spatial_distances'